# Implementing an Earley Parser (Chatty Version)

This notebook implements the same Earley parser as the notebook `01-Earley-Parser.ipynb`, but in addition it shows
how the parser proceeds:  Before the parser processes the set $Q_i$, it prints the next token.  Every *Earley item*
that is added by one of the operations *completion*, *prediction*, or *scanning* is printed together with the name of
the operation.  Finally, the sets $Q_i$ and $Q_{i+1}$ are printed.  Only the class `EarleyParser` and its methods
differ from the notebook `01-Earley-Parser.ipynb`.

## Setup and Imports

We load the OCaml port of `Larkita`.  The parsers generated by `Larkita` return the abstract syntax tree of their input as a nested
tuple.

In [ ]:
#use "../lib/nb.ml";;
load_lib "larkita.ml";;
load_lib "regex.ml"

The file `simple.g` contains a grammar describing arithmetic expressions that only use the operators `+` and `*`.

In [ ]:
cat "simple.g"

## Specification of the Grammar

`Larkita` allows us to define both lexical tokens and structural rules natively in a single *extended Backus-Naur
Form* (EBNF) string.  Below is the grammar that `Larkita` will use to parse our context-free grammar specifications
(like `simple.g`).  These specifications are written in the notation of `Larkita`, too:  The alternatives of a
rule are separated by `"|"`, and a rule is not terminated by a semicolon.
* The token `VARIABLE` recognizes a *syntactical variable*.  Syntactical variables start with a lower case letter.
* The token `RULE_NAME` recognizes the variable that is defined by a rule, i.e. a variable that is followed by a
  colon.  After a rule, a variable can either be the next item of this rule or the beginning of the next rule, and
  only the colon tells these cases apart.  As an LR(1) parser sees only one token ahead, the scanner has to make this
  distinction:  The *lookahead* `(?=[ \t]*:)` at the end of the regular expression checks that a colon follows
  without making it part of the token, and the priority `2` makes the scanner prefer `RULE_NAME` to `VARIABLE`.
* The token `TERMINAL` recognizes a *token*.  Tokens start with an uppercase variable.
* The token `LITERAL` recognizes a single character that is enclosed in double quotes.
* The string literals of an alternative form the head of the tuple that is returned for this alternative.  As the
  literals `":"` and `"|"` carry no information, the annotation `%ignore ":", "|"` removes them.  Then no literal is
  left and the name of the rule is used as head:  A rule yields the tuple `('rule', variable, body_1, ..., body_n)`,
  and a body yields the tuple `('body', item_1, ..., item_n)`.
* An alternative that consists of a single symbol returns the abstract syntax tree of this symbol.  Therefore, the
  rule `item` creates no tuple of its own.  A token is represented as a pair consisting of the name of the token and
  its text, for example `('LITERAL', '"+"')`.
* The `%ignore` directive tells the scanner to discard whitespace and newlines so they do not clutter our syntax tree.

In [ ]:
let meta_grammar = {|
start: rule+

rule: RULE_NAME ":" body ("|" body)*    %ignore ":", "|"

body: item*

item: VARIABLE
    | TERMINAL
    | LITERAL

RULE_NAME.2: /[a-z][a-z0-9_]*(?=[ \t]*:)/
VARIABLE:    /[a-z][a-z0-9_]*/
TERMINAL:    /[A-Z][A-Z0-9_]*/
LITERAL:     /"."/

%import common.WS
%ignore WS
|}

We generate the parser.  `Larkita` always generates a *canonical LR(1) parser* and writes a description of its states
into the file `grammar-of-grammars.out`.

In [ ]:
let grammar_parser = Larkita.create ~out_file:"grammar-of-grammars.out" meta_grammar

Let us check that Larkita was indeed able to generate a parser.

In [ ]:
let report = read_file "grammar-of-grammars.out" in
let rec find i = if String.sub report i 7 = "States\n" then i else find (i + 1) in
print_string (String.sub report 0 (find 0));
flush stdout

## Generating the Abstract Syntax Tree (AST)

The parser returns a nested tuple of the form `('start', rule_1, ..., rule_n)`.  Our custom `EarleyParser` expects the
grammar to be a list of rules, where each rule is a list of strings (e.g., `["expr"; "expr"; "+"; "product"]`).  The
function `symbol` converts a token into a string, taking care to strip the surrounding quote marks from `LITERAL`
tokens.  The function `ast_to_rules` drops the heads `'start'`, `'rule'`, and `'body'`:  For every body of a rule
defining the variable `a`, it creates the list that starts with `a` and continues with the items of the body, and
it applies `symbol` to all these tokens.

In [ ]:
let symbol (token : ast) : string =
  match token with
  (* Strip the surrounding double quotes (e.g., "\"+\"" -> "+") *)
  | Tok ("LITERAL", text) -> String.sub text 1 (String.length text - 2)
  | Tok (_, text) -> text
  | _ -> invalid_arg ("symbol: token expected, got " ^ Larkita.to_string token)

let ast_to_rules (ast : ast) : string list list =
  let rules = match ast with Node (_, rules) -> rules | _ -> invalid_arg "ast_to_rules: tuple expected" in
  List.concat_map
    (fun rule ->
      match rule with
      | Node (_, variable :: bodies) ->
          List.map
            (fun body ->
              match body with
              | Node (_, items) -> symbol variable :: List.map symbol items
              | _ -> invalid_arg "ast_to_rules: body expected")
            bodies
      | _ -> invalid_arg "ast_to_rules: rule expected")
    rules

The `parse` function reads the grammar file, parses it into a nested tuple, and converts it into our native nested
lists structure.

In [ ]:
let parse (file : string) : string list list =
  let grammar_str = read_file file in
  print_endline grammar_str;
  let ast = Larkita.parse grammar_parser grammar_str in
  ast_to_rules ast

In [ ]:
parse "simple.g"

## Earley's Algorithm

Given a context-free grammar $G = \langle V, \Sigma, R, s \rangle$ and a string $w = x_1x_2 \cdots x_n \in \Sigma^*$ of length $n$,
an *Earley item* is a pair of the form
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
such that
- $(a \rightarrow \alpha \beta) \in R\quad$  and
- $k \in \{0,1,\cdots,n\}$.

In Python, the class `EarleyItem` represents a single *Earley item*.  In OCaml, we define the module `EarleyItem`,
which contains the record type `t` with the following fields:
- `mVariable` is the variable $a$,
- `mAlpha` is $\alpha$,
- `mBeta` is $\beta$, and
- `mIndex` is $k$.

The function `make` creates an Earley item.  It corresponds to the constructor of the Python class.

Since we later have to store Earley items in sets, the Python class has to implement the methods `__eq__`, `__ne__`,
and `__hash__`.  In OCaml, this is not necessary: records are compared *structurally*, i.e. two records are equal iff
their fields are equal, so they can be stored in sets of type `FSet.t` immediately.  The function `to_string` corresponds
to the method `__repr__`.  It is installed as the printer of Earley items.

In [ ]:
module EarleyItem = struct
  type t = { mVariable : string; mAlpha : string list; mBeta : string list; mIndex : int }

  let make (variable : string) (alpha : string list) (beta : string list) (index : int) : t =
    { mVariable = variable; mAlpha = alpha; mBeta = beta; mIndex = index }

  let to_string (self : t) : string =
    let alphaStr = String.concat " " self.mAlpha in
    let betaStr  = String.concat " " self.mBeta in
    Printf.sprintf "<%s → %s • %s, %d>" self.mVariable alphaStr betaStr self.mIndex

  let pp ppf (self : t) = Format.pp_print_string ppf (to_string self)
end
;;
#install_printer EarleyItem.pp

Given an Earley item `self`, the function `isComplete` checks, whether the Earley item `self` has the form
$$\langle A \rightarrow \alpha \bullet, k \rangle,$$
i.e. whether the $\bullet$ is at the end of the grammar rule.

The Python version adds this function to the class `EarleyItem` via the assignment `EarleyItem.isComplete = isComplete`.
In OCaml, we *extend* the module `EarleyItem`: the new module `EarleyItem` *includes* everything from the old module
and adds the function `isComplete`.  Afterwards, the function is called as `EarleyItem.isComplete item`.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let isComplete (self : t) : bool = self.mBeta = []
end

The function `sameVar self c` checks, whether the item following the dot is the same as the variable
given as argument, i.e. `sameVar self c` returns `true` if `self` is an Earley item of the form
$$\langle a \rightarrow \alpha \bullet c\beta, k \rangle.$$

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let sameVar (self : t) (c : string) : bool =
    match self.mBeta with
    | b :: _ -> b = c
    | [] -> false
end

The function `scan self t` checks, whether the item following the dot matches the token `t`,
i.e. `scan self t` returns `true` if `self` is an Earley item of the form
$$\langle a \rightarrow \alpha \bullet T\beta, k \rangle.$$
The argument $T$ can either be the name of a token or a literal.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let scan (self : t) (t : string) : bool =
    match self.mBeta with
    | b :: _ -> b = t
    | [] -> false
end

Given an Earley item, this function returns the name of the variable following the dot.  If there is no variable
following the dot, the function returns `None`.  Its result has the type `string option`: it is either `None` or
`Some var`.  The function can distinguish variables from token names because variable names consist only of lower
case letters.  The function `is_lower` corresponds to Python's method `islower`: it checks that the string contains a
lower case letter, but no upper case letter.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let is_lower (s : string) : bool =
    String.exists (fun c -> 'a' <= c && c <= 'z') s && not (String.exists (fun c -> 'A' <= c && c <= 'Z') s)

  let nextVar (self : t) : string option =
    match self.mBeta with
    | var :: _ when var.[0] <> '"' && is_lower var -> Some var
    | _ -> None
end

The function `moveDot self` moves the $\bullet$ in the Earley item `self`, where `self` has the form
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
over the next variable, token, or literal in $\beta$.  It assumes that $\beta$ is not empty.

In [ ]:
module EarleyItem = struct
  include EarleyItem

  let moveDot (self : t) : t =
    match self.mBeta with
    | b :: rest -> make self.mVariable (self.mAlpha @ [ b ]) rest self.mIndex
    | [] -> invalid_arg "moveDot: the dot is already at the end"
end

The module `Grammar` represents a context free grammar.  It stores a list of the rules of the grammar.
Each grammar rule of the form
$$ a \rightarrow \beta $$
is stored as the list $[a] \mathbin{@} \beta$.  The start symbol is assumed to be the variable on the left hand side of
the first rule. To distinguish syntactical variables from tokens, variables contain only lower case letters,
while tokens either contain only upper case letters or they are literals, i.e. single characters like `+`, since
the function `symbol` has stripped the double quotes of the literals.

The functions `startItem` and `finishItem` call the function `startVar`, which is only defined later.  Therefore, the
module contains a *forward reference* `startVar_ref`, i.e. a mutable box that will later contain the function
`startVar`.  `!startVar_ref` is the function that is currently stored in the box.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

module Grammar = struct
  type t = { mRules : string list list }

  let make (rules : string list list) : t = { mRules = rules }

  let startVar_ref : (t -> string) ref = ref not_yet_defined
end

The function `startItem` returns the Earley item
$$ \langle\hat{s} \rightarrow \bullet s, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is the new variable `Start`.

In [ ]:
module Grammar = struct
  include Grammar

  let startItem (self : t) : EarleyItem.t = EarleyItem.make "Start" [] [ !startVar_ref self ] 0
end

The function `finishItem` returns the Earley item
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is a new variable.

In [ ]:
module Grammar = struct
  include Grammar

  let finishItem (self : t) : EarleyItem.t = EarleyItem.make "Start" [ !startVar_ref self ] [] 0
end

The function `startVar` returns the start variable of the grammar.  It is assumed that
the first rule grammar starts with the start variable of the grammar.

In [ ]:
module Grammar = struct
  include Grammar

  let startVar (self : t) : string = List.hd (List.hd self.mRules)
end
;;
Grammar.startVar_ref := Grammar.startVar

The module `EarleyParser` implements the [parsing algorithm of Jay Earley](https://en.wikipedia.org/wiki/Earley_parser).
Its record type `t` has the following fields:
- `mGrammar` is the grammar that is used to parse the given token string.
- `mString` is the array of tokens and literals that has to be parsed.

   As a hack, the first element of this array is the empty string.
   Therefore, `mString.(i)` is the $i^\textrm{th}$ token.
- `mStateList` is an array of sets of *Earley items*.  If $n$ is the length of the given token string
  (excluding the first element), then $Q_i = \texttt{mStateList.(i)}$.
  The idea is that the set $Q_i$ is the set of those *Earley items* that the parser could be in
  when it has read the tokens `mString.(1)`, $\cdots$,  `mString.(i)`.  $Q_0$ is initialized as follows:
  $$ Q_0 = \bigl\{\langle\hat{s} \rightarrow \bullet s, 0\rangle\bigr\}. $$
  Since the sets are changed by the algorithm, `mStateList` is an *array*, whose elements can be replaced.

  In order to print the *Earley items* in the order in which they are added, the sets $Q_i$ are stored as
  *lists* without duplicates:  The function `add` appends an item to such a list unless it is already an element.
  (The Python version uses dictionaries, since a dictionary remembers the order in which its keys have been
  inserted.)  The function `make` prints the list of tokens.

The *Earley items* are interpreted as follows: If we have
$$ \langle c \rightarrow \alpha \bullet \beta, k\rangle \in Q_i, $$
then we know the following:
- After having read the tokens `mString[:k+1]` the parser tries to parse the variable $c$
  in the token string `mString[k+1:]`.
- After having read the token string `mString[k+1:i+1]` the parser has already recognized $\alpha$
  and now needs to recognize $\beta$ in the token string `mString[i+1:]` in order to parse the variable $c$.

The function `make` creates a parser.  The method `parse` calls the methods `complete`, `predict`, and `scan`, which
are defined later, so the module contains forward references for them.

In [ ]:
module EarleyParser = struct
  type t = { mGrammar : Grammar.t; mString : string array; mStateList : EarleyItem.t list array }

  let make (grammar : Grammar.t) (tokenList : string list) : t =
    let mString    = Array.of_list ("" :: tokenList) in   (* hack so mString.(1) is the first token *)
    let mStateList = Array.make (List.length tokenList + 1) [] in
    mStateList.(0) <- [ Grammar.startItem grammar ];
    Printf.printf "Input: [%s]\n" (String.concat ", " (List.map Larkita.py_repr tokenList));
    { mGrammar = grammar; mString; mStateList }

  (* add item to the end of the list items, unless it is already there *)
  let add (items : EarleyItem.t list) (item : EarleyItem.t) : EarleyItem.t list =
    if List.mem item items then items else items @ [ item ]

  let print_items (items : EarleyItem.t list) : unit =
    List.iter (fun item -> print_endline (EarleyItem.to_string item)) items

  let complete_ref : (t -> int -> bool) ref = ref not_yet_defined
  let predict_ref  : (t -> int -> bool) ref = ref not_yet_defined
  let scan_ref     : (t -> int -> unit) ref = ref not_yet_defined
end

The method `parse` implements Earley's algorithm.  For all states 
$Q_1$, $\cdots$, $Q_n$ we proceed as follows:
- We apply the *completion* operation followed by the *prediction* operation.
  This is done until no more states are added to $Q_i$.  
  
  (The inner `while` loop is not necessary if the grammar does not contain $\varepsilon$-rules.)
- Finally, the *scanning* operation is applied to $Q_i$.  This operation adds
  items to the set $Q_{i+1}$.

After $Q_i$ has been computed, we proceed to process $Q_{i+1}$.
Parsing is successful iff
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle \in Q_n $$

Before the set $Q_i$ is processed, the method `parse` prints the next token.  After the scanning operation, it
prints the sets $Q_i$ and $Q_{i+1}$.

In [ ]:
module EarleyParser = struct
  include EarleyParser

  let parse (self : t) : unit =
    (* run Earley's algorithm *)
    let n = Array.length self.mString - 1 in   (* mString.(0) = "" *)
    for i = 0 to n do
      let next_token = if i + 1 <= n then self.mString.(i + 1) else "EOF" in
      print_endline (String.make 80 '_');
      Printf.printf "next token = %s\n" next_token;
      print_endline (String.make 80 '_');
      let rec loop () =
        let change = !complete_ref self i in
        let change = !predict_ref self i || change in
        if change then loop ()
      in
      loop ();
      !scan_ref self i;
      Printf.printf "\nQ%d:\n" i;
      print_items self.mStateList.(i);
      if i + 1 <= n then begin
        Printf.printf "\nQ%d:\n" (i + 1);
        print_items self.mStateList.(i + 1)
      end
    done;
    if List.mem (Grammar.finishItem self.mGrammar) self.mStateList.(n) then print_endline "Parsing successful!"
    else print_endline "Parsing failed!"
end

The method `complete(self, i)` applies the completion operation to the state $Q_i$:
If we have
 - $\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i$  and
 - $\langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j$,
then the parser tried to parse the variable $c$ after having read `mString[:j+1]`
and we know that 
$$ c \Rightarrow^* \texttt{mString[j+1:i+1]}, $$
i.e. the parser has recognized $c$ after having read `mString[j+1:i+1]`.
Therefore the parser should proceed to recognize $\delta$ in state $Q_i$.
Therefore we add the *Earley item* $\langle a \rightarrow \beta c \bullet \delta,k\rangle$ to the set $Q_i$:
$$\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i \wedge
  \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j \;\rightarrow\;
          Q_i := Q_i \cup \bigl\{ \langle a \rightarrow \beta c \bullet \delta, k\rangle \bigr\}
$$

The method prints every *Earley item* that it adds to $Q_i$.  Therefore, `newQi` only collects those
*Earley items* that are not yet elements of $Q_i$, and the loop stops as soon as `newQi` is empty.

In [ ]:
module EarleyParser = struct
  include EarleyParser

  let complete (self : t) (i : int) : bool =
    let change = ref false in
    let added  = ref true in
    while !added do
      added := false;
      let qi = self.mStateList.(i) in
      let newQi = ref [] in
      List.iter
        (fun item ->
          if EarleyItem.isComplete item then begin
            let c  = item.EarleyItem.mVariable in
            let qj = self.mStateList.(item.EarleyItem.mIndex) in
            List.iter
              (fun newItem ->
                if EarleyItem.sameVar newItem c then begin
                  let moved = EarleyItem.moveDot newItem in
                  if not (List.mem moved qi) then newQi := add !newQi moved
                end)
              qj
          end)
        qi;
      if !newQi <> [] then begin
        change := true;
        added  := true;
        print_endline "completion:";
        List.iter (fun item -> Printf.printf "%s added to Q%d\n" (EarleyItem.to_string item) i) !newQi;
        self.mStateList.(i) <- List.fold_left add qi !newQi
      end
    done;
    !change
end
;;
EarleyParser.complete_ref := EarleyParser.complete

The method `self.predict(i)` applies the prediction operation to the state $Q_i$: 
If $\langle a \rightarrow \beta \bullet c \delta, k \rangle \in Q_i$, then
the parser tries to recognize $c\delta$ after having read `mString[:i+1]`.  To this end
it has to parse $c$ in the string `mString[i+1:]`.
Therefore, if $c \rightarrow \gamma$ is a rule of our grammar,
we add the *Earley item* $\langle c \rightarrow \bullet \gamma, i\rangle$ to the set $Q_i$:
$$ \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_i 
       \wedge (c \rightarrow \gamma) \in R 
       \;\rightarrow\;
       Q_i := Q_i \cup\bigl\{ \langle c \rightarrow \bullet\gamma, i\rangle\bigr\}.
$$
As the right hand side $\gamma$ might start with a variable, the function uses a fix point iteration
until no more *Earley items* are added to $Q_i$.

The method prints every *Earley item* that it adds to $Q_i$.  Therefore, `newQi` only collects those
*Earley items* that are not yet elements of $Q_i$, and the loop stops as soon as `newQi` is empty.

In [ ]:
module EarleyParser = struct
  include EarleyParser

  let predict (self : t) (i : int) : bool =
    let change = ref false in
    let added  = ref true in
    while !added do
      added := false;
      let qi = self.mStateList.(i) in
      let newQi = ref [] in
      List.iter
        (fun item ->
          match EarleyItem.nextVar item with
          | Some c ->
              List.iter
                (fun rule ->
                  if c = List.hd rule then begin
                    let newItem = EarleyItem.make c [] (List.tl rule) i in
                    if not (List.mem newItem qi) then newQi := add !newQi newItem
                  end)
                self.mGrammar.Grammar.mRules
          | None -> ())
        qi;
      if !newQi <> [] then begin
        change := true;
        added  := true;
        print_endline "prediction:";
        List.iter (fun item -> Printf.printf "%s added to Q%d\n" (EarleyItem.to_string item) i) !newQi;
        self.mStateList.(i) <- List.fold_left add qi !newQi
      end
    done;
    !change
end
;;
EarleyParser.predict_ref := EarleyParser.predict

The function `self.scan(i)` applies the scanning operation to the state $Q_i$.

If $\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i$ and $T$ is a token,
then the parser tries to recognize the right hand side of the grammar rule
$$ a \rightarrow \beta T \gamma$$ 
and after having read `mString[k+1:i+1]` it has already recognized  $\beta$.
If we now have `mString[i+1] == a`, then the parser still has to recognize $\gamma$ in `mString[i+2:]`.
Therefore, the *Earley object* $\langle a \rightarrow \beta T \bullet \gamma, k\rangle$ is added to
the set $Q_{i+1}$:
$$\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i \wedge x_{i+1} = T
       \;\rightarrow\;
       Q_{i+1} := Q_{i+1} \cup \bigl\{ \langle a \rightarrow \beta T \bullet \gamma, k\rangle \bigr\}
$$

The method prints every *Earley item* that it adds to $Q_{i+1}$.

In [ ]:
module EarleyParser = struct
  include EarleyParser

  let scan (self : t) (i : int) : unit =
    let qi = self.mStateList.(i) in
    let n  = Array.length self.mString - 1 in   (* remember mString.(0) = "" *)
    if i + 1 <= n then begin
      let a = self.mString.(i + 1) in
      print_endline "scanning:";
      List.iter
        (fun item ->
          if EarleyItem.scan item a then begin
            let moved = EarleyItem.moveDot item in
            Printf.printf "%s added to Q%d\n" (EarleyItem.to_string moved) (i + 1);
            self.mStateList.(i + 1) <- add self.mStateList.(i + 1) moved
          end)
        qi
    end
end
;;
EarleyParser.scan_ref := EarleyParser.scan

The function `tokenize` transforms the string `s` that is to be parsed into a list of tokens. See below for an example.
The regular expression contains five groups.  The module `Re` is loaded at the beginning of the notebook.  Therefore, we use the function `Re.findall_groups`, which returns the list
of the strings matched by the groups for every match.  Exactly one of these strings is not empty.

In [ ]:
let tokenize (s : string) : string list =
  (* Transform the string s into a list of tokens.  The string s
     is supposed to represent an arithmetic expression. *)
  let lexSpec = {|([ \t]+)        |  # blanks and tabs
                  ([1-9][0-9]*|0) |  # number
                  ([()])          |  # parentheses
                  ([-+*/])        |  # arithmetical operators
                  (.)                # unrecognized character
               |}
  in
  let tokenList = Re.findall_groups ~flags:Re.[VERBOSE] lexSpec s in
  List.filter_map
    (function
      | [ ws; number; parenthesis; operator; error ] ->
          if ws <> "" then None                       (* skip blanks and tabs *)
          else if number <> "" then Some "NUMBER"
          else if parenthesis <> "" then Some parenthesis
          else if operator <> "" then Some operator
          else Some (Printf.sprintf "ERROR(%s)" error)
      | _ -> assert false)
    tokenList

In [ ]:
tokenize "1 + 2 * 3"

The function `test` takes two arguments.
- `file` is the name of a file containing a grammar,
- `word` is a string that should be parsed.

`word` is first tokenized. Then the grammar is parsed by the parser generated by `Larkita` into a nested tuple, converted by our `ast_to_rules` function, and passed directly to the function `Grammar.make`.  Finally, `word` is parsed using *Earley's algorithm*.

In [ ]:
let test (file : string) (word : string) : unit =
  let grammarStr = read_file file in
  print_endline grammarStr;
  (* Parse the grammar into a nested tuple and convert it into a list of rules *)
  let ast       = Larkita.parse grammar_parser grammarStr in
  let rules     = ast_to_rules ast in
  let grammar   = Grammar.make rules in
  let tokenList = tokenize word in
  let ep        = EarleyParser.make grammar tokenList in
  EarleyParser.parse ep

In [ ]:
test "simple.g" "1 + 2 * 3"